# Dataset exploration

Loads the processed SemEval-2014 Task 4 records and reports the statistics the models and the report rely on: size, label balance, aspect frequency, review length and the sentences that carry no aspect annotation.

## Setup

Run this on a Colab GPU runtime (*Runtime → Change runtime type → GPU*). The cell clones the repository, installs the dependencies and checks the environment. Colab keeps its own `torch`, so it is left as it is.

In [ ]:
import os

if not os.path.isdir("aspect-based-sentiment-analysis"):
    !git clone -q https://github.com/Shivam-Shukla0/aspect-based-sentiment-analysis.git
%cd aspect-based-sentiment-analysis
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps
!python -m scripts.verify_environment --require-gpu

## Data

The official SemEval-2014 Task 4 files are distributed through the ILSP Metashare repository and need an account and acceptance of its licence (see `data/README.md`). Download them once and keep them in Google Drive; this cell copies them into `data/raw/` and builds the processed record files. Nothing in `data/` is committed to Git.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/absa/semeval2014")  # change to where you stored the files
FILES = {
    "Laptop_Train_v2.xml": ("laptop", "train"),
    "Restaurants_Train_v2.xml": ("restaurant", "train"),
    "Laptops_Test_Gold.xml": ("laptop", "test"),
    "Restaurants_Test_Gold.xml": ("restaurant", "test"),
}
Path("data/raw").mkdir(parents=True, exist_ok=True)
for name, (domain, split) in FILES.items():
    !cp "{DRIVE_DIR / name}" data/raw/
    !python -m scripts.build_processed data/raw/{name} --domain {domain} --split {split}

## Size and label balance

The `conflict` label is rare, and the models are trained and scored without it (see the report for the protocol).

In [ ]:
import json
from collections import Counter

from absa.data import aspect_frequency, class_balance, review_lengths


def load(domain, split):
    with open(f"data/processed/{domain}_{split}.jsonl", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle]


for domain in ("laptop", "restaurant"):
    for split in ("train", "test"):
        records = load(domain, split)
        print(f"{domain:10s} {split:5s} {len(records):5d} records  {class_balance(records)}")

## Aspect frequency and review length

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, domain in zip(axes, ("laptop", "restaurant")):
    top = aspect_frequency(load(domain, "train"), top_n=10)
    ax.barh([a for a, _ in top][::-1], [c for _, c in top][::-1])
    ax.set_title(f"Most frequent aspect terms ({domain}, train)")
plt.tight_layout()
plt.show()

for domain in ("laptop", "restaurant"):
    lengths = review_lengths(load(domain, "train"))
    print(f"{domain}: review length min {min(lengths)}, max {max(lengths)}, mean {sum(lengths) / len(lengths):.1f} characters")

## Sentences without an aspect term

These are kept in separate files rather than dropped, because they matter for end-to-end evaluation.

In [ ]:
for domain in ("laptop", "restaurant"):
    for split in ("train", "test"):
        with open(f"data/processed/{domain}_{split}_no_aspect.jsonl", encoding="utf-8") as handle:
            print(domain, split, sum(1 for _ in handle), "sentences without an aspect term")